## Noon Offers Scrapping

In [1]:
import selenium 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- Offers List --
Offers = []


## -- Scraping --
driver = webdriver.Chrome(options=options)

driver.get(
    "https://www.noon.com/egypt-en/electronics-and-mobiles/eg-november26-clearance-yfs/?f%5Bdeal_tag%5D%5B%5D=mega-deal&sort%5Bby%5D=popularity&sort%5Bdir%5D=desc"
)

time.sleep(5)

page = 1

while True:

    page_products = 0

    ## -- Product Cards --
    cards = driver.find_elements(
        By.CSS_SELECTOR, 'div[data-qa="plp-product-box"]'
    )

    for card in cards:

        ## -- Product Info --
        try:
            Product = card.find_element(
                By.CSS_SELECTOR, 'a[href*="/p/"]'
            )

            Link = Product.get_attribute("href")

            Name = card.find_element(
                By.CSS_SELECTOR, 'h2[data-qa="plp-product-box-name"]'
            ).text

            Image = card.find_element(
                By.CSS_SELECTOR, 'img[class*="productImage"]'
            ).get_attribute("src")

        except NoSuchElementException:
            continue

        ## -- NEW PRICE --
        try:
            New_Price = card.find_element(
                By.CSS_SELECTOR,
                '[data-qa="plp-product-box-price"] .sellingPrice strong'
            ).text

        except NoSuchElementException:
            New_Price = "Product Not Available"

        ## -- OLD PRICE --
        try:
            Old_Price = card.find_element(
                By.CSS_SELECTOR,
                '[data-qa="plp-product-box-price"] .strikeThrough'
            ).text

        except NoSuchElementException:
            Old_Price = None

        ## -- Stock --
        try:
            Stock = card.find_element(
                By.CSS_SELECTOR, '[data-inventory-status]'
            ).get_attribute("data-inventory-status")

        except NoSuchElementException:
            Stock = "Product Not Available"

        ## -- Append Data --
        Offers.append({
            "Name": Name,
            "Link": Link,
            "Image": Image,
            "New_Price": New_Price,
            "Old_Price": Old_Price,
            "Stock": Stock
        })

        page_products += 1

    ## -- Page Summary --
    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Offers)}"
    )

    ## -- Next Page --
    next_buttons = driver.find_elements(
        By.CSS_SELECTOR, "a[aria-label='Next']"
    )

    if not next_buttons:
        print("No Next page. Scraping finished.")
        break

    next_url = next_buttons[0].get_attribute("href")

    if not next_url or next_url == driver.current_url:
        print("Invalid or unchanged Next URL.")
        break

    print("Navigating to:", next_url)

    driver.get(next_url)

    page += 1
    time.sleep(5)


## -- Final Result --
print(f"\nScraping completed. Total products: {len(Offers)}")

driver.quit()